In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Separem columnes categoriques de les numèriques
id_cols_categorical = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in id_cols_categorical:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

# Check
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Apliquem OneHotEncodera var. categoriques i Standard Scaler a var. numèriques
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)


logreg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',  # corregeix el desbalanceig 11%/89%
        max_iter=1000,            # el default (100) no convergeix amb 264 features
        random_state=42
    ))
])

logreg_pipeline.fit(X_train, y_train)

print("Logistic Regression pipeline trained.")
print(f"Features after one-hot: {logreg_pipeline.named_steps['preprocessor'].transform(X_train.head(1)).shape[1]}")

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# Probabilitats predites per la classe positiva (readmissió <30 dies)
y_pred_proba = logreg_pipeline.predict_proba(X_test)[:, 1]

# Calculem mètriques
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"=== Logistic Regression — Test set ===")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"Baseline PR-AUC (random): {y_test.mean():.4f}")

# Provem diversos thresholds: el 0.5 default rarament és l'òptim en context clínic
for threshold in [0.3, 0.4, 0.5]:
    y_pred = (y_pred_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n--- Threshold = {threshold} ---")
    print(f"Confusion matrix:")
    print(f"  TN={cm[0,0]:>6}   FP={cm[0,1]:>6}")
    print(f"  FN={cm[1,0]:>6}   TP={cm[1,1]:>6}")
    print(classification_report(y_test, y_pred, target_names=['Not readmitted', 'Readmitted'], digits=3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Segon model per contrastar amb la regressió logística
# max_depth=10 limita la profunditat per evitar overfitting amb 264 features
# n_jobs=-1 paral·lelitza l'entrenament a tots els cores disponibles
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
pr_auc_rf = average_precision_score(y_test, y_pred_proba_rf)

print(f"=== Random Forest — Test set ===")
print(f"ROC-AUC: {roc_auc_rf:.4f}")
print(f"PR-AUC:  {pr_auc_rf:.4f}")

for threshold in [0.3, 0.4, 0.5]:
    y_pred_rf = (y_pred_proba_rf >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_rf)
    print(f"\n--- Threshold = {threshold} ---")
    print(f"  TN={cm[0,0]:>6}   FP={cm[0,1]:>6}")
    print(f"  FN={cm[1,0]:>6}   TP={cm[1,1]:>6}")
    print(classification_report(y_test, y_pred_rf, target_names=['Not readmitted', 'Readmitted'], digits=3))

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

# ROC i PR curves dels dos models en una sola figura per comparar
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC: TPR vs FPR. La diagonal és un classificador aleatori
for name, proba in [('Logistic Regression', y_pred_proba), ('Random Forest', y_pred_proba_rf)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')

# PR: precision vs recall. La baseline és la proporció de positius (~11.6%)
for name, proba in [('Logistic Regression', y_pred_proba), ('Random Forest', y_pred_proba_rf)]:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(recall, precision, label=f'{name} (AP = {ap:.3f})', linewidth=2)

axes[1].axhline(y_test.mean(), color='k', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('../figures/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to figures/roc_pr_curves.png")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Threshold 0.4 com a compromís entre recall i precision per al context clínic
threshold = 0.4
y_pred_final = (y_pred_proba >= threshold).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=['Not readmitted', 'Readmitted <30d'],
    cmap='Blues',
    ax=ax,
    colorbar=False
)
ax.set_title(f'Logistic Regression — Confusion Matrix (threshold = {threshold})')
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import shap

# Mostrem 1000 files del test: amb 20k files SHAP triga massa i el resultat és igual d'estable
np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=1000, replace=False)
X_test_sample = X_test.loc[sample_idx]

# Apliquem el preprocessor manualment per accedir a les features ja codificades
X_test_transformed = logreg_pipeline.named_steps['preprocessor'].transform(X_test_sample)
feature_names = logreg_pipeline.named_steps['preprocessor'].get_feature_names_out()

# LinearExplainer és la versió optimitzada de SHAP per a models lineals
explainer = shap.LinearExplainer(
    logreg_pipeline.named_steps['classifier'],
    X_test_transformed
)
shap_values = explainer.shap_values(X_test_transformed)

# Top 15 features per impacte mitjà absolut
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.tight_layout()
plt.savefig('../figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score


# 5-fold CV per validar que el resultat del test set no és fruit d'un split de sort
# Stratified manté la proporció de positius (~11%) constant a cada fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_logreg = cross_val_score(
    logreg_pipeline, X_train, y_train,
    cv=cv, scoring='roc_auc', n_jobs=-1
)

print(f"Logistic Regression — 5-fold CV ROC-AUC")
print(f"Scores: {cv_scores_logreg.round(4)}")
print(f"Mean:   {cv_scores_logreg.mean():.4f}")
print(f"Std:    {cv_scores_logreg.std():.4f}")
print(f"Test:   {roc_auc:.4f}  (for reference)")